# FlashVSR End-to-End Video Super-Resolution on Trainium 2

This notebook demonstrates the full FlashVSR pipeline on trn2.3xlarge:
- **LQ Projection**: Generates per-token conditioning from low-quality input
- **DiT Denoising**: Streaming chunks via NxDI ModelBuilder (TP=4)
- **TCDecoder**: Latent-to-RGB with HBM state persistence
- **Color Correction**: AdaIN alignment with LQ reference

**Requirements:**
- Instance: trn2.3xlarge (LNC=2, 4 logical NeuronCores)
- AMI: Deep Learning AMI Neuron (Ubuntu 24.04) 20260502 or later
- Venv: `/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/bin/activate`

**Expected results:**
- 85 output frames at 768x1280 resolution
- End-to-end throughput: ~10 FPS
- All models co-resident in HBM (no model swapping)

## 1. Environment Setup

In [1]:
import os
import sys
import time
import gc
import torch
import concurrent.futures
import numpy as np

os.environ["NEURON_FUSE_SOFTMAX"] = "1"

# Patch ThreadPoolExecutor for single-process NxDI operation
original_tpe_init = concurrent.futures.ThreadPoolExecutor.__init__
def patched_tpe_init(self, *args, **kwargs):
    kwargs["max_workers"] = 1
    original_tpe_init(self, *args, **kwargs)
concurrent.futures.ThreadPoolExecutor.__init__ = patched_tpe_init

import torch_neuronx
print(f"PyTorch: {torch.__version__}")
print(f"torch-neuronx: {torch_neuronx.__version__}")
print(f"Device: {os.popen('neuron-ls 2>/dev/null | head -3').read().strip()}")

PyTorch: 2.9.1+cu128
torch-neuronx: 2.9.0.2.13.26312+8e870898
Device: instance-type: trn2.3xlarge
instance-id: i-0c9a2547029fc84a6
logical-neuroncore-config: 2


## 2. Configuration

Set paths to weights, compiled models, and input video.

The pipeline has four compiled components:
- **DiT first chunk** (f=6 latent frames): `flashvsr_first_tp4/`
- **DiT stream chunk** (f=2 latent frames): `flashvsr_stream_tp4/`
- **LQ Projection** (torch_neuronx.trace): `lq_proj/lq_proj_T89.pt`
- **TCDecoder** (NxDI ModelBuilder, HBM states): `tcdecoder_notebook_tp4/`

In [2]:
# Paths -- adjust these for your setup
WEIGHTS_DIR = os.path.expanduser("~/FlashVSR-v1.1")
COMPILED_DIR = os.path.expanduser("~/flash_vsr/compiled")
TCDECODER_DIR = os.path.expanduser("~/compiled/tcdecoder_notebook_tp4")
PROMPT_PATH = os.path.expanduser("~/flash_vsr/posi_prompt.pth")
INPUT_VIDEO = os.path.expanduser("~/flash_vsr/example0_cropped_192x320.mp4")
OUTPUT_DIR = os.path.expanduser("~/flash_vsr/notebook_output")

# Hardware config
TP_DEGREE = 4
HEIGHT = 768
WIDTH = 1280

# Verify paths exist
for name, path in [("Weights", WEIGHTS_DIR), ("DiT first", f"{COMPILED_DIR}/flashvsr_first_tp4"),
                    ("DiT stream", f"{COMPILED_DIR}/flashvsr_stream_tp4"),
                    ("TCDecoder", TCDECODER_DIR), ("Prompt", PROMPT_PATH),
                    ("Input video", INPUT_VIDEO)]:
    exists = os.path.exists(path)
    print(f"  {name}: {path} {'[OK]' if exists else '[MISSING]'}")
    assert exists, f"Missing: {path}"

  Weights: /home/ubuntu/FlashVSR-v1.1 [OK]
  DiT first: /home/ubuntu/flash_vsr/compiled/flashvsr_first_tp4 [OK]
  DiT stream: /home/ubuntu/flash_vsr/compiled/flashvsr_stream_tp4 [OK]
  TCDecoder: /home/ubuntu/compiled/tcdecoder_notebook_tp4 [OK]
  Prompt: /home/ubuntu/flash_vsr/posi_prompt.pth [OK]
  Input video: /home/ubuntu/flash_vsr/example0_cropped_192x320.mp4 [OK]


## 3. Add Source to Path

In [3]:
# Add the FlashVSR contrib source as a package
# The src/ directory contains __init__.py and uses relative imports
FLASHVSR_ROOT = os.path.abspath(os.path.join(os.path.dirname("."), ".."))
if not os.path.exists(os.path.join(FLASHVSR_ROOT, "src")):
    FLASHVSR_ROOT = os.path.abspath("..")
sys.path.insert(0, FLASHVSR_ROOT)
print(f"Package root: {FLASHVSR_ROOT}")

from src.modeling_flashvsr import (
    FlashVSRApplication,
    FlashVSRInferenceConfig,
    precompute_freqs_cis_3d,
    build_rope_for_grid,
    HEAD_DIM, DIM, NUM_HEADS, PATCH_T, PATCH_H, PATCH_W, LCSA_WIN,
)
from src.tcdecoder import (
    TCDecoderApplication, TCDecoderConfig, TCPixelShuffle3d,
    INPUT_CHANNELS, NUM_MEM_BLOCKS, decode_video_nxdi,
)
from src.pipeline import (
    neuron_dit_forward, prepare_input_tensor,
    color_correct_wavelet, tensor2video, save_video,
)
from neuronx_distributed_inference.models.config import NeuronConfig

print(f"DiT: DIM={DIM}, HEADS={NUM_HEADS}, HEAD_DIM={HEAD_DIM}")
print(f"TCDecoder: INPUT_CHANNELS={INPUT_CHANNELS}, MEM_BLOCKS={NUM_MEM_BLOCKS}")

Package root: /home/ubuntu/test_notebook


DiT: DIM=1536, HEADS=12, HEAD_DIM=128
TCDecoder: INPUT_CHANNELS=784, MEM_BLOCKS=9


/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:100: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise_mm_baseline_shard_hidden: No module named 'neuronxcc.nki._private.blockwise_mm'
  warnings.warn(f"Warning: {error}")
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise_mm_bwd: No module named 'neuronxcc.nki._private.blockwise_mm_bwd'
  warnings.warn(f"Warning: {error}")
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise_mm_bwd_

## 4. Load All Models (Co-resident in HBM)

All four compiled components are loaded onto the same 4 NeuronCores:
- DiT first chunk: ~7.5 GB
- DiT stream chunk: ~7.5 GB  
- TCDecoder: ~378 MB
- LQ Projection: ~1.2 GB

Total HBM usage: ~16.6 GB out of 96 GB available.

In [4]:
timings = {}
t_load_start = time.time()

# --- Load LQ Projection (torch_neuronx.trace) ---
print("Loading LQ Projection...")
t0 = time.time()
lq_proj_path = os.path.join(COMPILED_DIR, "lq_proj", "lq_proj_T89.pt")
lq_proj_model = torch.jit.load(lq_proj_path)
timings['lq_load'] = time.time() - t0
print(f"  Loaded in {timings['lq_load']:.1f}s")

# --- Load DiT (first chunk, f=6) ---
print("Loading DiT (first chunk)...")
t0 = time.time()
neuron_config = NeuronConfig(
    tp_degree=TP_DEGREE,
    torch_dtype=torch.bfloat16,
    batch_size=1,
    save_sharded_checkpoint=True,
)
dit_first_config = FlashVSRInferenceConfig(
    neuron_config=neuron_config,
    attn_mode="first",
    height=HEIGHT,
    width=WIDTH,
)
dit_first_app = FlashVSRApplication(model_path=WEIGHTS_DIR, config=dit_first_config)
dit_first_app.load(os.path.join(COMPILED_DIR, "flashvsr_first_tp4"))
timings['dit_first_load'] = time.time() - t0
print(f"  Loaded in {timings['dit_first_load']:.1f}s")

# --- Load DiT (stream chunk, f=2) ---
print("Loading DiT (stream chunk)...")
t0 = time.time()
dit_stream_config = FlashVSRInferenceConfig(
    neuron_config=neuron_config,
    attn_mode="stream",
    height=HEIGHT,
    width=WIDTH,
)
dit_stream_app = FlashVSRApplication(model_path=WEIGHTS_DIR, config=dit_stream_config)
dit_stream_app.load(os.path.join(COMPILED_DIR, "flashvsr_stream_tp4"))
timings['dit_stream_load'] = time.time() - t0
print(f"  Loaded in {timings['dit_stream_load']:.1f}s")

# --- Load TCDecoder (NxDI, HBM state persistence) ---
print("Loading TCDecoder...")
t0 = time.time()
tcd_neuron_config = NeuronConfig(
    tp_degree=TP_DEGREE,
    torch_dtype=torch.bfloat16,
    batch_size=1,
)
tcd_config = TCDecoderConfig(neuron_config=tcd_neuron_config, height=HEIGHT, width=WIDTH)
tcd_app = TCDecoderApplication(weights_dir=WEIGHTS_DIR, config=tcd_config)
tcd_app.load(TCDECODER_DIR)
timings['tcdecoder_load'] = time.time() - t0
print(f"  Loaded in {timings['tcdecoder_load']:.1f}s")

# --- Precompute RoPE ---
base_freqs = precompute_freqs_cis_3d(HEAD_DIM)

# --- Load prompt embedding ---
prompt_emb = torch.load(PROMPT_PATH, map_location="cpu")
if prompt_emb.dim() == 2:
    prompt_emb = prompt_emb.unsqueeze(0)
prompt_emb = prompt_emb.to(dtype=torch.bfloat16)

timings['total_load'] = time.time() - t_load_start
print(f"\nAll models loaded in {timings['total_load']:.1f}s")
print(f"All 4 components co-resident on {TP_DEGREE} NeuronCores")

Loading LQ Projection...
  Loaded in 13.5s
Loading DiT (first chunk)...


Neuron: Loading presharded checkpoints for ranks: 0...3


Neuron: Finished weights loading in 32.17717863899816 seconds


Neuron: Warming up the model.


2026-May-26 23:09:02.0761 16587:16659 [3] int nccl_net_ofi_create_plugin(nccl_net_ofi_plugin_t**):219 CCOM WARN NET/OFI Failed to initialize rdma protocol
2026-May-26 23:09:02.0764 16587:16659 [3] int nccl_net_ofi_create_plugin(nccl_net_ofi_plugin_t**):354 CCOM WARN NET/OFI aws-ofi-nccl initialization failed
2026-May-26 23:09:02.0767 16587:16659 [3] ncclResult_t nccl_net_ofi_init_no_atexit_fini_v6(ncclDebugLogger_t):183 CCOM WARN NET/OFI Initializing plugin failed
2026-May-26 23:09:02.0769 16587:16659 [3] net_plugin.cc:97 CCOM WARN OFI plugin initNet() failed is EFA enabled?


...

Neuron: Finished weights loading in 4.995344078000926 seconds


Neuron: Warming up the model.


Neuron: Warmup completed in 0.6243040561676025 seconds.


  Loaded in 5.7s
Loading TCDecoder...


  Loaded in 1.5s

All models loaded in 54.9s
All 4 components co-resident on 4 NeuronCores


## 5. Prepare Input Video

The input video is bicubic-upscaled to the target resolution and formatted as `(1, C, F, H, W)` in `[-1, 1]`.

In [5]:
print(f"Loading input: {INPUT_VIDEO}")
LQ_video, tH, tW, num_frames, fps = prepare_input_tensor(
    INPUT_VIDEO, scale=4, dtype=torch.bfloat16, device="cpu"
)
print(f"  Shape: {LQ_video.shape}")
print(f"  Frames: {num_frames} (8n+1 format)")
print(f"  Resolution: {tH}x{tW}")
print(f"  FPS: {fps}")
print(f"  Dtype: {LQ_video.dtype}")

Loading input: /home/ubuntu/flash_vsr/example0_cropped_192x320.mp4


/home/ubuntu/test_notebook/src/pipeline.py:116: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  t = torch.from_numpy(np.asarray(img, np.uint8)).to(


  Shape: torch.Size([1, 3, 89, 768, 1280])
  Frames: 89 (8n+1 format)
  Resolution: 768x1280
  FPS: 30
  Dtype: torch.bfloat16


## 6. Stage 1: LQ Projection

Processes all LQ frames in a single pass to produce per-token conditioning residuals.
These guide the DiT denoising to preserve content from the input.

In [6]:
lat_h = HEIGHT // 8
lat_w = WIDTH // 8
tokens_per_frame = (tH // 16) * (tW // 16)
first_chunk_tokens = 6 * tokens_per_frame
stream_chunk_tokens = 2 * tokens_per_frame

print(f"Running LQ Projection on {LQ_video.shape[2]} frames...")
lq_input = LQ_video.to(dtype=torch.bfloat16)

with torch.no_grad():
    # Warmup
    _ = lq_proj_model(lq_input)
    
    # Timed run
    t0 = time.time()
    all_lq_tokens = lq_proj_model(lq_input)
    timings['lq_proj'] = time.time() - t0

print(f"  Output shape: {all_lq_tokens.shape}")
print(f"  Tokens per frame: {tokens_per_frame}")
print(f"  Time: {timings['lq_proj']*1000:.0f} ms")

Running LQ Projection on 89 frames...


  Output shape: torch.Size([1, 84480, 1536])
  Tokens per frame: 3840
  Time: 854 ms


## 7. Stage 2: DiT Streaming Denoising

The DiT processes latent frames in streaming chunks:
- **First chunk** (f=6): 6 latent frames → generates initial 24 output frames
- **Stream chunks** (f=2 each): 2 latent frames → 8 output frames per chunk, with temporal overlap

FlashVSR uses single-step DMD denoising (timestep=1000).

In [7]:
# Prepare noise latents
num_latent_frames = (num_frames - 1) // 4
noise = torch.randn(1, 16, num_latent_frames, lat_h, lat_w, dtype=torch.bfloat16)
latents = noise

# Calculate number of chunks
process_total_num = (num_frames - 1) // 8 - 2
print(f"DiT streaming: {process_total_num} chunks (1 first + {process_total_num-1} stream)")
print(f"  Latent frames: {num_latent_frames}")
print(f"  Latent shape: {noise.shape}")
print()

# Warmup DiT (one forward pass each)
print("Warming up DiT...")
with torch.no_grad():
    warmup_first = latents[:, :, :6, :, :]
    lq_r = all_lq_tokens[:, :first_chunk_tokens, :] if all_lq_tokens is not None else None
    _ = neuron_dit_forward(dit_first_app, base_freqs, warmup_first, prompt_emb, tH, tW, 0, lq_r)
    
    warmup_stream = latents[:, :, 4:6, :, :]
    lq_r_s = all_lq_tokens[:, first_chunk_tokens:first_chunk_tokens+stream_chunk_tokens, :] if all_lq_tokens is not None else None
    _ = neuron_dit_forward(dit_stream_app, base_freqs, warmup_stream, prompt_emb, tH, tW, 1, lq_r_s)
print("  Warmup complete")
print()

# Timed DiT inference
latents_total = []
chunk_times = []

t_dit_start = time.time()
with torch.no_grad():
    for cur_process_idx in range(process_total_num):
        # Select current chunk latents
        if cur_process_idx == 0:
            cur_latents = latents[:, :, :6, :, :]
        else:
            cur_latents = latents[:, :, 4 + cur_process_idx * 2 : 6 + cur_process_idx * 2, :, :]

        # Get LQ residual for this chunk
        lq_residual = None
        if all_lq_tokens is not None:
            if cur_process_idx == 0:
                lq_residual = all_lq_tokens[:, :first_chunk_tokens, :]
            else:
                offset = first_chunk_tokens + (cur_process_idx - 1) * stream_chunk_tokens
                lq_residual = all_lq_tokens[:, offset:offset + stream_chunk_tokens, :]

        # Select DiT model
        active_app = dit_first_app if cur_process_idx == 0 else dit_stream_app

        # Forward pass
        t_chunk = time.time()
        noise_pred = neuron_dit_forward(
            active_app, base_freqs, cur_latents, prompt_emb, tH, tW,
            cur_process_idx, lq_residual_0=lq_residual,
        )
        chunk_time = time.time() - t_chunk
        chunk_times.append(chunk_time)

        # One-step denoising (DMD)
        cur_latents = cur_latents - noise_pred
        latents_total.append(cur_latents)

        chunk_type = "first" if cur_process_idx == 0 else "stream"
        print(f"  Chunk {cur_process_idx} ({chunk_type}, f={cur_latents.shape[2]}): {chunk_time*1000:.0f} ms")

timings['dit_total'] = time.time() - t_dit_start
latents_out = torch.cat(latents_total, dim=2)

print(f"\nDiT complete: {timings['dit_total']:.2f}s")
print(f"  First chunk: {chunk_times[0]*1000:.0f} ms")
print(f"  Stream chunks ({len(chunk_times)-1}x): {sum(chunk_times[1:])*1000:.0f} ms (avg {np.mean(chunk_times[1:])*1000:.0f} ms)")
print(f"  Output latent shape: {latents_out.shape}")

DiT streaming: 9 chunks (1 first + 8 stream)
  Latent frames: 22
  Latent shape: torch.Size([1, 16, 22, 96, 160])

Warming up DiT...


  Warmup complete



  Chunk 0 (first, f=6): 1692 ms


  Chunk 1 (stream, f=2): 415 ms


  Chunk 2 (stream, f=2): 410 ms


...

  Chunk 4 (stream, f=2): 406 ms


  Chunk 5 (stream, f=2): 406 ms


  Chunk 6 (stream, f=2): 407 ms


  Chunk 7 (stream, f=2): 407 ms


  Chunk 8 (stream, f=2): 407 ms

DiT complete: 4.96s
  First chunk: 1692 ms
  Stream chunks (8x): 3265 ms (avg 408 ms)
  Output latent shape: torch.Size([1, 16, 22, 96, 160])


## 8. Stage 3: TCDecoder (Latent → RGB)

The TCDecoder converts DiT latent frames to full-resolution RGB using:
- HBM state persistence (`input_output_aliases`) — 9 MemBlock states stay in device memory
- Sequential processing — each call produces 4 output frames
- Output reshape trick — prevents TP sharding of temporal dimension

In [8]:
tc_pixel_shuffle = TCPixelShuffle3d(4, 8, 8)

# The number of LQ conditioning frames must match latent count after pixel shuffle.
# TCPixelShuffle3d(ff=4) groups 4 temporal frames into channels, so:
#   LQ_cur_idx = latents_T * pixel_shuffle_temporal_factor
latents_T = latents_out.shape[2]
LQ_cur_idx = latents_T * 4  # pixel shuffle temporal factor
# Clamp to available frames
LQ_cur_idx = min(LQ_cur_idx, LQ_video.shape[2])

print(f"TCDecoder: processing {latents_T} latent frames")
print(f"  LQ reference frames: {LQ_cur_idx}")
print(f"  After pixel shuffle: T={LQ_cur_idx//4}")

# Warmup TCDecoder (reset + 2 calls)
tcd_app.reset_states()
warmup_x = torch.randn(1, INPUT_CHANNELS, lat_h, lat_w, dtype=torch.bfloat16)
with torch.no_grad():
    _ = tcd_app(warmup_x)
    _ = tcd_app(warmup_x)

# Timed decode
t0 = time.time()
frames = decode_video_nxdi(
    tcd_app,
    latents_out.transpose(1, 2),  # (1, C, T, H, W) -> (1, T, C, H, W)
    LQ_video[:, :, :LQ_cur_idx, :, :],
    tc_pixel_shuffle,
    frames_to_trim=3,
)
timings['tcdecoder'] = time.time() - t0

print(f"  Output shape: {frames.shape}")
print(f"  Output frames: {frames.shape[2]}")
print(f"  Resolution: {frames.shape[3]}x{frames.shape[4]}")
print(f"  Time: {timings['tcdecoder']:.2f}s")
print(f"  Per-frame: {timings['tcdecoder']/frames.shape[2]*1000:.1f} ms")

TCDecoder: processing 22 latent frames
  LQ reference frames: 88
  After pixel shuffle: T=22


  Output shape: torch.Size([1, 3, 85, 768, 1280])
  Output frames: 85
  Resolution: 768x1280
  Time: 2.43s
  Per-frame: 28.6 ms


## 9. Stage 4: Color Correction

AdaIN color correction aligns the output color distribution with the LQ reference.
This is a CPU operation and adds minimal overhead.

In [9]:
import torch.nn.functional as F

# --- Neuron-accelerated AdaIN color correction ---
class NeuronAdaIN(torch.nn.Module):
    """AdaIN color correction, traceable for torch_neuronx.trace()."""
    def __init__(self):
        super().__init__()
        self.eps = 1e-5

    def forward(self, content, style):
        N, C = content.shape[:2]
        content_flat = content.view(N, C, -1)
        content_mean = content_flat.mean(dim=2, keepdim=True)
        content_var = content_flat.var(dim=2, unbiased=False, keepdim=True) + self.eps
        content_std = content_var.sqrt()
        style_flat = style.view(N, C, -1)
        style_mean = style_flat.mean(dim=2, keepdim=True)
        style_var = style_flat.var(dim=2, unbiased=False, keepdim=True) + self.eps
        style_std = style_var.sqrt()
        normalized = (content_flat - content_mean) / content_std
        result = normalized * style_std + style_mean
        return torch.clamp(result.view_as(content), -1.0, 1.0)

ADAIN_BATCH = 16
adain_compiled_path = os.path.expanduser("~/compiled/adain_neuron.pt")

if os.path.exists(adain_compiled_path):
    print("Loading pre-compiled Neuron AdaIN...")
    adain_model = torch.jit.load(adain_compiled_path)
else:
    print(f"Compiling Neuron AdaIN (batch={ADAIN_BATCH}, {HEIGHT}x{WIDTH})...")
    adain_module = NeuronAdaIN().eval()
    example_content = torch.randn(ADAIN_BATCH, 3, HEIGHT, WIDTH, dtype=torch.bfloat16)
    example_style = torch.randn(ADAIN_BATCH, 3, HEIGHT, WIDTH, dtype=torch.bfloat16)
    adain_model = torch_neuronx.trace(
        adain_module, (example_content, example_style),
        compiler_args=['--auto-cast', 'matmult', '-O1'],
    )
    torch.jit.save(adain_model, adain_compiled_path)
    print(f"  Saved to {adain_compiled_path}")

# Warmup
warmup_c = torch.randn(ADAIN_BATCH, 3, HEIGHT, WIDTH, dtype=torch.bfloat16)
warmup_s = torch.randn(ADAIN_BATCH, 3, HEIGHT, WIDTH, dtype=torch.bfloat16)
with torch.no_grad():
    _ = adain_model(warmup_c, warmup_s)

# Run color correction on Neuron
print("Applying Neuron AdaIN color correction...")
t0 = time.time()

n_output_frames = frames.shape[2]
lq_frames_for_cc = min(n_output_frames, LQ_video.shape[2])

# Resize LQ to match output resolution (skip if already at target size)
lq_raw = LQ_video[:, :, :lq_frames_for_cc, :, :].reshape(-1, 3, tH, tW)
if tH == HEIGHT and tW == WIDTH:
    lq_resized = lq_raw  # already at target resolution
else:
    lq_resized = F.interpolate(
        lq_raw, size=(HEIGHT, WIDTH), mode='bilinear', align_corners=False,
    )  # (lq_frames_for_cc, 3, H, W)

# Process in batches of ADAIN_BATCH frames
hq_frames = frames[0, :, :lq_frames_for_cc].permute(1, 0, 2, 3)  # (T, 3, H, W)
corrected_batches = []
for start in range(0, lq_frames_for_cc, ADAIN_BATCH):
    end = min(start + ADAIN_BATCH, lq_frames_for_cc)
    batch_hq = hq_frames[start:end]
    batch_lq = lq_resized[start:end]
    # Pad to ADAIN_BATCH if needed
    actual_count = batch_hq.shape[0]
    if actual_count < ADAIN_BATCH:
        # Repeat-pad to fill batch (handles case where pad_n > actual_count)
        repeats = (ADAIN_BATCH + actual_count - 1) // actual_count
        batch_hq = batch_hq.repeat(repeats, 1, 1, 1)[:ADAIN_BATCH]
        batch_lq = batch_lq.repeat(repeats, 1, 1, 1)[:ADAIN_BATCH]
    with torch.no_grad():
        out = adain_model(batch_hq, batch_lq)
    corrected_batches.append(out[:actual_count])

corrected_all = torch.cat(corrected_batches, dim=0)  # (T, 3, H, W)
frames_corrected = corrected_all.permute(1, 0, 2, 3).unsqueeze(0)  # (1, 3, T, H, W)

# Append uncorrected frames if any
if lq_frames_for_cc < n_output_frames:
    frames_corrected = torch.cat([frames_corrected, frames[:, :, lq_frames_for_cc:, :, :]], dim=2)

timings['color_correction'] = time.time() - t0

print(f"  Time: {timings['color_correction']*1000:.0f} ms (Neuron-accelerated)")
print(f"  Output range: [{frames_corrected.min():.2f}, {frames_corrected.max():.2f}]")

Loading pre-compiled Neuron AdaIN...


Applying Neuron AdaIN color correction...


  Time: 407 ms (Neuron-accelerated)
  Output range: [-1.00, 1.00]


## 10. Save Output Video

In [10]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, "output.mp4")

pil_frames = tensor2video(frames_corrected[0])
save_video(pil_frames, output_path, fps=fps)

output_size = os.path.getsize(output_path) / 1024 / 1024
print(f"Saved: {output_path}")
print(f"  Frames: {len(pil_frames)}")
print(f"  Resolution: {pil_frames[0].size[0]}x{pil_frames[0].size[1]}")
print(f"  FPS: {fps}")
print(f"  File size: {output_size:.1f} MB")

Saved: /home/ubuntu/flash_vsr/notebook_output/output.mp4
  Frames: 85
  Resolution: 1280x768
  FPS: 30
  File size: 1.5 MB


## 11. Performance Summary

In [11]:
# Compute throughput (both with and without color correction)
neuron_time = timings['lq_proj'] + timings['dit_total'] + timings['tcdecoder']
e2e_time = neuron_time + timings['color_correction']
output_frames = frames_corrected.shape[2]
neuron_fps = output_frames / neuron_time
e2e_fps = output_frames / e2e_time

print("=" * 64)
print("FlashVSR E2E Benchmark Summary")
print("=" * 64)
print(f"  Instance:           trn2.3xlarge (LNC=2, TP={TP_DEGREE})")
print(f"  Input:              {num_frames} frames at {tH//4}x{tW//4}")
print(f"  Output:             {output_frames} frames at {HEIGHT}x{WIDTH}")
print(f"  Scale:              4x")
print()
print(f"  --- Pipeline Timing ---")
print(f"  LQ Projection:      {timings['lq_proj']*1000:.0f} ms")
print(f"  DiT Denoising:      {timings['dit_total']*1000:.0f} ms ({process_total_num} chunks)")
print(f"    First chunk (f=6):  {chunk_times[0]*1000:.0f} ms")
print(f"    Stream chunks (f=2): {sum(chunk_times[1:])*1000:.0f} ms ({len(chunk_times)-1} x {np.mean(chunk_times[1:])*1000:.0f} ms avg)")
print(f"  TCDecoder:          {timings['tcdecoder']*1000:.0f} ms (HBM state persistence)")
print(f"  Color correction:   {timings['color_correction']*1000:.0f} ms (Neuron AdaIN)")
print(f"  ---")
print(f"  Neuron pipeline:    {neuron_time:.2f}s -> {neuron_fps:.1f} FPS")
print(f"  Full E2E:           {e2e_time:.2f}s -> {e2e_fps:.1f} FPS (incl. color correction)")
print()
print(f"  --- Model Loading (one-time) ---")
print(f"  LQ Projection:      {timings['lq_load']:.1f}s")
print(f"  DiT (first):        {timings['dit_first_load']:.1f}s")
print(f"  DiT (stream):       {timings['dit_stream_load']:.1f}s")
print(f"  TCDecoder:          {timings['tcdecoder_load']:.1f}s")
print(f"  Total load:         {timings['total_load']:.1f}s")
print("=" * 64)

FlashVSR E2E Benchmark Summary
  Instance:           trn2.3xlarge (LNC=2, TP=4)
  Input:              89 frames at 192x320
  Output:             85 frames at 768x1280
  Scale:              4x

  --- Pipeline Timing ---
  LQ Projection:      854 ms
  DiT Denoising:      4961 ms (9 chunks)
    First chunk (f=6):  1692 ms
    Stream chunks (f=2): 3265 ms (8 x 408 ms avg)
  TCDecoder:          2432 ms (HBM state persistence)
  Color correction:   407 ms (Neuron AdaIN)
  ---
  Neuron pipeline:    8.25s -> 10.3 FPS
  Full E2E:           8.65s -> 9.8 FPS (incl. color correction)

  --- Model Loading (one-time) ---
  LQ Projection:      13.5s
  DiT (first):        34.3s
  DiT (stream):       5.7s
  TCDecoder:          1.5s
  Total load:         54.9s


In [12]:
# Restore ThreadPoolExecutor
concurrent.futures.ThreadPoolExecutor.__init__ = original_tpe_init